# **Assignment 1**

---
## 📚 **Introduction**
This template notebook is designed to guide you through Assignment 1 of the LLM System course. Follow the steps to set up your environment, implement the required CUDA kernels, and test your code.

### 🚀 **Goal of the Assignment**
You will implement high-performance CUDA kernels for tensor operations and integrate them with the MiniTorch framework. You will implement low-level operators in CUDA C++ and connect them to Python through the CUDA backend. This assignment focuses on parallel computing concepts and GPU acceleration techniques.

---

## ⚙️ **Environment Setup**
First, ensure that you have changed the runtime to **T4 GPU**. Run the following commands to set up your environment.

In [2]:
# Clone the starter code repository
!git clone https://github.com/llmsystem/llmsys_f25_hw1.git
%cd llmsys_f25_hw1

Cloning into 'llmsys_f25_hw1'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 253 (delta 100), reused 82 (delta 82), pack-reused 118 (from 1)
Receiving objects: 100% (253/253), 1.02 MiB | 37.45 MiB/s, done.
Resolving deltas: 100% (125/125), done.
/content/llmsys_f25_hw1


In [3]:
# Install dependencies
!python -m pip install -r requirements.txt
!python -m pip install -r requirements.extra.txt
!python -m pip install -Ue .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 70.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.7 MB/s

In [4]:
!pip install hypothesis
!pip install pycuda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.1/538.1 kB 37.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 47.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.5 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2025.1.2-cp312-cp312-linux_x86_64.whl size=659050 sha256=385a0ad96e546d808d2bfe9d17afdd1c90e70543e353ab7c01eeab3af4b21e87
  Stored in directory: /root/.cache/pip/wheels/d5/36/f3/ac5f09d768cad3fa15d5a3449bdfe65c3de58e69d036c73228
Successfully built pycuda


In [5]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Tue Dec 16 13:00:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8       

In [ ]:
import torch; print(torch.version.cuda)

12.8


---

## 🔧 **CUDA Kernel Compilation**
You will need to compile the CUDA kernels for this assignment. Run the following command to create the necessary directory and compile the CUDA files.

---

In [ ]:
%cd llmsys_f25_hw1

[Errno 2] No such file or directory: 'llmsys_f25_hw1'
/content/llmsys_f25_hw1


In [21]:
# Compile CUDA kernels
!mkdir -p minitorch/cuda_kernels
!rm -rf minitorch/cuda_kernels/*
!nvcc -o minitorch/cuda_kernels/combine.so --shared src/combine.cu -Xcompiler -fPIC -arch=sm_75 --ptxas-options=-v
!ls -l minitorch/cuda_kernels/*

src/combine.cu(462): warning #177-D: variable "a_batch_stride" was declared but never referenced
      int a_batch_stride = a_shape[0] > 1 ? a_strides[0] : 0;
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

src/combine.cu(463): warning #177-D: variable "b_batch_stride" was declared but never referenced
      int b_batch_stride = b_shape[0] > 1 ? b_strides[0] : 0;
          ^

ptxas info    : 63 bytes gmem, 16 bytes cmem[4]
ptxas info    : Compiling entry function '_Z20MatrixMultiplyKernelPfPKiS1_S_S1_S1_S_S1_S1_' for 'sm_75'
ptxas info    : Function properties for _Z20MatrixMultiplyKernelPfPKiS1_S_S1_S1_S_S1_S1_
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 41 registers, 8192 bytes smem, 424 bytes cmem[0]
ptxas info    : Compiling entry function '_Z12reduceKernelPfPiS0_iS_S0_S0_ifii' for 'sm_75'
ptxas info    : Function properties for _Z12reduceKernelPfPiS0_iS_S0_S0_ifii
    80 bytes stack frame, 0 by

In [7]:
import minitorch
import minitorch.testing as testing
from minitorch.cuda_kernel_ops import CudaKernelOps
# 创建一个小的测试张量
backend = minitorch.TensorBackend(CudaKernelOps)
t = minitorch.tensor([1.0, -2.0, 3.0], backend=backend)
result = testing.MathTestVariable.neg1(t)
print('Success:', result)

Success: 
[-1.000000 2.000000 -3.000000]


## 📋 **Assignment Sections**

### 🧮 **Problem 1: Map Operation CUDA Kernel + Integration (15 points)**
**Goal:** Implement the CUDA kernel for element-wise map operations and integrate it with the MiniTorch framework.

The map operation applies a unary function to every element of an input tensor, producing a new tensor with the same shape. For example, applying `f(x) = x²` to tensor `[1, 2, 3]` yields `[1, 4, 9]`.

🔧 **Instructions:**
1. Navigate to `src/combine.cu`.
2. Locate the placeholders marked with `BEGIN ASSIGN2_1` and `END ASSIGN2_1`.
3. Implement the `mapKernel` function.

**Key Points:**
- Each thread should process one element of the output tensor
- Use thread and block indices to calculate the global thread ID
- Ensure proper bounds checking to avoid out-of-bounds memory access
- Consider the stride-based indexing for multidimensional tensors

**Testing:**
Run the following command to test your implementation.

```python
!python -m pytest -l -v -k "cuda_one_args"
```

---

In [8]:
# Problem 1: Map Operation CUDA Kernel Tests

# TODO: Implement the mapKernel function in src/combine.cu
# Make sure to recompile CUDA kernels before testing:
# !nvcc -o minitorch/cuda_kernels/combine.so --shared src/combine.cu -Xcompiler -fPIC

!python -m pytest -l -v -k "cuda_one_args"


============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: /content/llmsys_f25_hw1
configfile: setup.cfg
plugins: hypothesis-6.148.7, langsmith-0.4.58, typeguard-4.4.4, anyio-4.12.0
collected 26 items / 24 deselected / 2 selected                                

tests/test_tensor_general.py::test_cuda_one_args[cuda-fn0] PASSED        [ 50%]
tests/test_tensor_general.py::test_cuda_one_args[cuda-fn1] PASSED        [100%]

======================= 2 passed, 24 deselected in 3.91s =======================


---

### 🚀 **Problem 2: Zip Operation CUDA Kernel + Integration (20 points)**

**Goal:** Implement the CUDA kernel for element-wise zip operations and integrate it with the framework.

This operation applies a binary function to corresponding elements from two input tensors, producing a new tensor with the same shape. For example, applying addition `f(x,y) = x + y` to tensors `[1, 2, 3]` and `[4, 5, 6]` yields `[5, 7, 9]`.

#### **Part A: Implement zipKernel (15 points)**

🔧 **Instructions:**
1. Navigate to `src/combine.cu`.
2. Locate the placeholders marked with `BEGIN ASSIGN2_2` and `END ASSIGN2_2`.
3. Implement the `zipKernel` function.

**Key Points:**
- Each thread processes one element from each input tensor
- Both input tensors should have the same shape or be broadcastable
- Handle stride-based indexing for both input tensors
- Ensure proper bounds checking

#### **Part B: Integrate Zip Operation (5 points)**

🔧 **Instructions:**
1. Navigate to `minitorch/cuda_kernel_ops.py`.
2. Locate the placeholders marked with `BEGIN ASSIGN2_2_INTEGRATION` and `END ASSIGN2_2_INTEGRATION`.
3. Implement the `zip` function in the `CudaKernelOps` class.

**Testing:**
```bash
!python -m pytest -l -v -k "cuda_two_args"
```


In [16]:
!nvcc -o minitorch/cuda_kernels/combine.so --shared src/combine.cu -Xcompiler -fPIC -arch=sm_75 --ptxas-options=-v

src/combine.cu(462): warning #177-D: variable "a_batch_stride" was declared but never referenced
      int a_batch_stride = a_shape[0] > 1 ? a_strides[0] : 0;
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

src/combine.cu(463): warning #177-D: variable "b_batch_stride" was declared but never referenced
      int b_batch_stride = b_shape[0] > 1 ? b_strides[0] : 0;
          ^

ptxas info    : 63 bytes gmem, 16 bytes cmem[4]
ptxas info    : Compiling entry function '_Z20MatrixMultiplyKernelPfPKiS1_S_S1_S1_S_S1_S1_' for 'sm_75'
ptxas info    : Function properties for _Z20MatrixMultiplyKernelPfPKiS1_S_S1_S1_S_S1_S1_
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 41 registers, 8192 bytes smem, 424 bytes cmem[0]
ptxas info    : Compiling entry function '_Z12reduceKernelPfPiS0_iS_S0_S0_ifii' for 'sm_75'
ptxas info    : Function properties for _Z12reduceKernelPfPiS0_iS_S0_S0_ifii
    80 bytes stack frame, 0 by

In [17]:
# Problem 2: Zip Operation CUDA Kernel Tests

# TODO:
# 1. Implement the zipKernel function in src/combine.cu
# 2. Implement the zip integration in minitorch/cuda_kernel_ops.py
# Make sure to recompile CUDA kernels before testing:
# !nvcc -o minitorch/cuda_kernels/combine.so --shared src/combine.cu -Xcompiler -fPIC

!python -m pytest -l -v -k "cuda_two_args"


============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: /content/llmsys_f25_hw1
configfile: setup.cfg
plugins: hypothesis-6.148.7, langsmith-0.4.58, typeguard-4.4.4, anyio-4.12.0
collected 26 items / 19 deselected / 7 selected                                

tests/test_tensor_general.py::test_cuda_two_args[cuda-fn0] PASSED        [ 14%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn1] PASSED        [ 28%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn2] PASSED        [ 42%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn3] PASSED        [ 57%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn4] PASSED        [ 71%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn5] PASSED        [ 85%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn6] PASSED        [100%]

===========

### 🧠 **Problem 3: Reduce Operation CUDA Kernel + Integration (20 points)**

**Goal:** Implement the CUDA kernel for reduction operations and integrate it with the framework.

This operation aggregates elements along a specified dimension of a tensor using a binary function, producing a tensor with reduced dimensionality. For example, reducing tensor `[[1, 2, 3], [4, 5, 6]]` along dimension 1 with sum yields `[6, 15]`.

#### **Part A: Implement reduceKernel (15 points)**

🔧 **Instructions:**
1. Navigate to `src/combine.cu`.
2. Locate the placeholders marked with `BEGIN ASSIGN2_3` and `END ASSIGN2_3`.
3. Implement the `reduceKernel` function.

**Key Points - Basic Reduction:**
- A simple way to parallelize the reduce function is to have every reduced element in the output calculated individually in each block
- Each block takes care of computing one output element
- It's important to think about how to calculate the step across the data to be reduced based on `reduce_dim` and `strides`

#### **Part B: Integrate Reduce Operation (5 points)**

🔧 **Instructions:**
1. Navigate to `minitorch/cuda_kernel_ops.py`.
2. Locate the placeholders marked with `BEGIN ASSIGN2_3_INTEGRATION` and `END ASSIGN2_3_INTEGRATION`.
3. Implement the `reduce` function in the `CudaKernelOps` class.

**Testing:**
```bash
!python -m pytest -l -v -k "cuda_reduce"
```

---

In [18]:
# Problem 3: Reduce Operation CUDA Kernel Tests

# TODO:
# 1. Implement the reduceKernel function in src/combine.cu
# 2. Implement the reduce integration in minitorch/cuda_kernel_ops.py
# Make sure to recompile CUDA kernels before testing:
# !nvcc -o minitorch/cuda_kernels/combine.so --shared src/combine.cu -Xcompiler -fPIC

!python -m pytest -l -v -k "cuda_reduce"

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: /content/llmsys_f25_hw1
configfile: setup.cfg
plugins: hypothesis-6.148.7, langsmith-0.4.58, typeguard-4.4.4, anyio-4.12.0
collected 26 items / 23 deselected / 3 selected                                

tests/test_tensor_general.py::test_cuda_reduce_sum_practice1[cuda] PASSED [ 33%]
tests/test_tensor_general.py::test_cuda_reduce_sum_practice2[cuda] PASSED [ 66%]
tests/test_tensor_general.py::test_cuda_reduce_sum_practice3[cuda] PASSED [100%]

======================= 3 passed, 23 deselected in 2.26s =======================


### 📈 **Problem 4: Matrix Multiplication CUDA Kernel + Integration (25 points)**

**Goal:** Implement the CUDA kernel for matrix multiplication and integrate it with the framework.

This is one of the most important operations in deep learning and offers significant opportunities for optimization.

#### **Part A: Implement MatrixMultiplyKernel (20 points)**

🔧 **Instructions:**
1. Navigate to `src/combine.cu`.
2. Locate the placeholders marked with `BEGIN ASSIGN2_4` and `END ASSIGN2_4`.
3. Implement the `MatrixMultiplyKernel` function.

**Key Points - Simple Parallelization:**
- A simple way to parallelize matrix multiplication is to have every element in the output matrix calculated individually in each thread
- Each thread computes one output element by performing the dot product of the corresponding row and column
- Use proper indexing to handle 2D thread blocks and memory access patterns

#### **Part B: Integrate Matrix Multiplication (5 points)**

🔧 **Instructions:**
1. Navigate to `minitorch/cuda_kernel_ops.py`.
2. Locate the placeholders marked with `BEGIN ASSIGN2_4_INTEGRATION` and `END ASSIGN2_4_INTEGRATION`.
3. Implement the `matrix_multiply` function in the `CudaKernelOps` class.

**Testing:**
```bash
!python -m pytest -l -v -k "cuda_matmul"
```

---

In [23]:
# Problem 4: Matrix Multiplication CUDA Kernel Tests

# TODO:
# 1. Implement the MatrixMultiplyKernel function in src/combine.cu
# 2. Implement the matrix_multiply integration in minitorch/cuda_kernel_ops.py
# Make sure to recompile CUDA kernels before testing:
# !nvcc -o minitorch/cuda_kernels/combine.so --shared src/combine.cu -Xcompiler -fPIC

!python -m pytest -l -v -k "cuda_matmul"

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: /content/llmsys_f25_hw1
configfile: setup.cfg
plugins: hypothesis-6.148.7, langsmith-0.4.58, typeguard-4.4.4, anyio-4.12.0
collected 26 items / 13 deselected / 13 selected                               

tests/test_tensor_general.py::test_cuda_matmul_numpy_eq[cuda-2-2-2] PASSED [  7%]
tests/test_tensor_general.py::test_cuda_matmul_numpy_eq[cuda-33-33-33] PASSED [ 15%]
tests/test_tensor_general.py::test_cuda_matmul_numpy_eq[cuda-16-16-16] PASSED [ 23%]
tests/test_tensor_general.py::test_cuda_matmul_numpy_eq[cuda-8-8-8] PASSED [ 30%]
tests/test_tensor_general.py::test_cuda_matmul_numpy_eq[cuda-1-2-3] PASSED [ 38%]
tests/test_tensor_general.py::test_cuda_matmul_numpy_eq[cuda-3-4-5] PASSED [ 46%]
tests/test_tensor_general.py::test_cuda_matmul_numpy_eq[cuda-5-4-3] PASSED

### 🎯 **Problem 5: Final Integration Test (5 points)**

**Goal:** Verify that all CUDA kernels work together correctly with comprehensive test cases.

After correctly implementing all functions in Problems 1-4, you should be able to pass all CUDA tests. This integration test includes more comprehensive test cases than the individual problem tests.

**Testing:**
Run the following command to test all CUDA implementations together:

```bash
!python -m pytest -l -v -k "cuda"
```

**Note:** If you pass the previous problem tests but fail here, please review your implementations for edge cases and ensure proper integration between kernels.

---


In [24]:
# Problem 5: Final Integration Test

# Run comprehensive CUDA tests to verify all implementations work together
!python -m pytest -l -v -k "cuda"


============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: /content/llmsys_f25_hw1
configfile: setup.cfg
plugins: hypothesis-6.148.7, langsmith-0.4.58, typeguard-4.4.4, anyio-4.12.0
collected 26 items                                                             

tests/test_tensor_general.py::test_create[cuda] PASSED                   [  3%]
tests/test_tensor_general.py::test_cuda_one_args[cuda-fn0] PASSED        [  7%]
tests/test_tensor_general.py::test_cuda_one_args[cuda-fn1] PASSED        [ 11%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn0] PASSED        [ 15%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn1] PASSED        [ 19%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn2] PASSED        [ 23%]
tests/test_tensor_general.py::test_cuda_two_args[cuda-fn3] PASSED        [ 26%]
tests/test_t

---

### 💾 **Submit Your Assignment: Create a ZIP File for Submission**

Run the following code to create a `llmsys_f25_hw1.zip` file, which you can download and upload to Canvas:


---

### 📋 **Instructions for Submission:**
1. **Run the cell below.**  
   - This will generate a `llmsys_f25_hw1.zip` file containing your entire project.
2. **Click the download link** that appears after the cell finishes running.
3. **Upload the downloaded ZIP file to Canvas.**



In [26]:
%cd ..

/content


In [27]:
import shutil

# Define the directory to zip
dir_to_zip = "llmsys_f25_hw1"

# Create a zip file
output_filename = f"{dir_to_zip}.zip"
shutil.make_archive(dir_to_zip, 'zip', dir_to_zip)

# Provide a download link
from google.colab import files
files.download(output_filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>